# 8. Reports and comparison

> **Install for this tutorial:** `pip install "scope-profiler[pproc]"`
> (interactive charts in the HTML report). The `site` command also needs
> `scope-profiler[site]` (PyYAML), which `pproc` already includes.

A profile is most useful next to another one: the same code before and after a
change, two job sizes, yesterday's run and today's. This notebook takes the
command-line tools that work on finished profile files:

1. `scope-profiler inspect` — summarize one or more runs
2. `scope-profiler diff` — compare two runs region by region
3. `scope-profiler report` — write a self-contained HTML report
4. `scope-profiler check` — fail a CI job when a run regressed
5. `scope-profiler site` — publish a catalogue of runs as a website

To have something to compare, the first step records three runs of a toy
solver: a baseline, a candidate whose `solve` has become twice as slow, and a
"fixed" version that is back to baseline speed.

In [ ]:
import os
import tempfile

# Work in a scratch directory so the tutorial leaves no files behind.
os.chdir(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))

## Three runs

`time.sleep` stands in for real work, as in the other tutorials, so the timings
are stable on any machine. Each run gets a `label`, which is the name `diff`,
`inspect` and the report use for it.

In [ ]:
import time

from scope_profiler import ProfileManager


def simulate(path, label, solve_cost=0.004, steps=20):
    """Record one run of the toy solver."""
    with ProfileManager.session(file_path=path, label=label, verbose=False):
        with ProfileManager.region("setup"):
            time.sleep(0.02)
        for _ in range(steps):
            with ProfileManager.region("timestep"):
                with ProfileManager.region("assemble"):
                    time.sleep(0.002)
                with ProfileManager.region("solve"):
                    time.sleep(solve_cost)


simulate("baseline.h5", "v1 (baseline)")
simulate("candidate.h5", "v2 (slower solve)", solve_cost=0.008)
simulate("fixed.h5", "v3 (fixed)")

Everything below is a command-line tool, so the next cell defines a small helper
that runs `scope-profiler` in a subprocess and shows the command, its output
and its exit code. In a terminal you would type the same command directly. (The
helper uses `python -m scope_profiler`, which is the same program as the
`scope-profiler` executable.)

In [ ]:
import os
import shlex
import subprocess
import sys


def sp(command):
    """Run `scope-profiler <command>` and print the command, output and exit code."""
    env = dict(os.environ, COLUMNS="100")  # the width of the tables
    args = [sys.executable, "-m", "scope_profiler", *shlex.split(command)]
    result = subprocess.run(args, check=False, capture_output=True, text=True, env=env)
    print("$ scope-profiler", command)
    print(result.stdout + result.stderr)
    print("exit code:", result.returncode)

## Summarize with `inspect`

`inspect` prints a run's metadata and its region statistics — the same table
as `results.print_summary()`. `--regions-only` and `--metadata-only` pick one
half; `--sort`, `--columns`, `--include`, `--exclude` and `--ranks` shape the
table. It also takes several files or a glob such as `'run_*.h5'`, printing one
summary per file.

In [ ]:
sp("inspect baseline.h5 --regions-only --sort total")

## Compare with `diff`

`diff a.h5 b.h5` lines up the regions of two runs, with `a` as the baseline, and
prints the change in each. `--metric` picks the statistic (`total` by default;
also `avg`, `min`, `max`, `p50`, `p95`, `p99`, `imbalance` or `calls`),
`--sort` the row order (`delta`, `pct` or `name`) and `--threshold PCT` hides
regions that changed by less than that many percent. It always exits with
code 0 — it reports, it does not judge.

In [ ]:
sp("diff baseline.h5 candidate.h5")

`solve` is +100% and everything that contains it (`timestep`, the session)
grows with it, while `assemble` and `setup` are unchanged. Per-call averages,
and only the regions that moved by more than 5%:

In [ ]:
sp("diff baseline.h5 candidate.h5 --metric avg --threshold 5")

The same comparison is available in Python as `scope_profiler.diff.diff_rows()`,
which returns one dict per region:

In [ ]:
from scope_profiler import load
from scope_profiler.diff import diff_rows

rows = diff_rows(load("baseline.h5"), load("candidate.h5"), metric="total")
for row in rows[:3]:
    print(
        f"{row['name']:<24} {row['a']:.3f} s -> {row['b']:.3f} s ({row['pct']:+.0f}%)"
    )

## An HTML report

`report` writes one self-contained HTML file: headline numbers, findings, ranked
hotspots, the region table with its call tree, per-rank load balance and,
with `scope-profiler[pproc]`, interactive charts. Metadata is always included.
`-o` names the file (default `report.html`). Given **one** run you get a full
report; given **several**, the first is the baseline of a comparison report
that shows what changed in each run, and a full report for every run is
written next to it as `<output>-<index>-<label>.html`.

The charts embed the Plotly runtime, which makes a report about 5 MB whatever
the profile. `--charts-cdn` loads Plotly from a CDN instead — about 30 times
smaller, at the price of needing a network connection to draw the charts
(the tables never do). `--no-charts` leaves the charts out altogether.

In [ ]:
sp("report baseline.h5 -o report.html")
sp("report baseline.h5 -o report_cdn.html --charts-cdn")
sp("report baseline.h5 candidate.h5 fixed.h5 -o comparison.html --charts-cdn")

In [ ]:
from pathlib import Path

for path in sorted(Path(".").glob("*.html")):
    print(f"{path.name:<40} {path.stat().st_size / 1e3:9.0f} kB")

### Opening a report

A report is an ordinary HTML file, so any browser opens it. Add `--show` to open
it as soon as it is written, or open it yourself:

```bash
scope-profiler report baseline.h5 candidate.h5 -o comparison.html --show
open comparison.html          # macOS; `xdg-open` on Linux
```

or from Python, with `webbrowser.open(Path("comparison.html").resolve().as_uri())`.
On a cluster, copy the file to your machine — it needs nothing else.

`scope-profiler run my_script.py -o report.html` skips the HDF5 file and writes
the report directly, since the extension picks the format. From Python the
function behind all of this is `scope_profiler.html_report.create_html_report()`.
To list several reports on one page, `scope-profiler catalogue` writes a small
index:

In [ ]:
sp("catalogue comparison.html report_cdn.html -o index.html --title 'Solver reports'")

## Gate a regression with `check`

`check` is `diff` with a verdict, meant for CI. It prints the comparison table
and exits with code **1** when any region got more than `--max-regression`
percent worse in the chosen metric (the default budget is 0%, so any slowdown
fails). `--include` / `--exclude` restrict which regions count, `--metric`
chooses the statistic (for example `p95` for tail latency, or `imbalance`
for MPI balance) and `--fail-on-new` also fails on regions that did not exist in
the baseline.

Timings of a real program wobble from run to run, so give the budget some
headroom. 50% is plenty here, where the regression is +100%:

In [ ]:
sp("check baseline.h5 candidate.h5 --max-regression 50")

The slowed-down candidate fails and names the regions over budget. The fixed
version is back within budget and passes:

In [ ]:
sp("check baseline.h5 fixed.h5 --max-regression 50")

In a CI job the exit code is all that is needed — the step simply fails:

```bash
scope-profiler run -o candidate.h5 benchmark.py
scope-profiler check baseline.h5 candidate.h5 --max-regression 10 --exclude 'setup.*'
```

The Python spelling is `scope_profiler.diff.check_files()`, which prints the
same report and returns the exit code (`0` or `1`):

In [ ]:
from scope_profiler.diff import check_files

with open(os.devnull, "w") as devnull:
    status = check_files(
        "baseline.h5", "candidate.h5", max_regression=50, stream=devnull
    )
print("check status:", status)

## A catalogue website with `site`

For a history of runs — nightly jobs, scaling studies, one case per code
version — standalone reports get unwieldy. `scope-profiler site` reads a YAML
manifest that lists *cases* (sets of runs shown together) and writes the pages
and chart data of a catalogue website into an Astro/Starlight project, whose
build turns them into a static site you can publish on GitHub Pages. Reading the
manifest needs PyYAML (the `site` extra) and the charts need `pproc`.

The [`scope-profiler-catalog`](https://github.com/max-models/scope-profiler-catalog)
repository is a template for such a site. The manifest is small:

In [ ]:
manifest = """
title: Solver profiling history
description: Three versions of the toy solver.
base: /solver-profiles
cases:
  - id: solver-versions
    title: Solver versions
    scaling: none
    runs:
      - {path: baseline.h5, label: v1}
      - {path: candidate.h5, label: v2}
      - {path: fixed.h5, label: v3}
"""
with open("manifest.yaml", "w") as handle:
    handle.write(manifest)

sp("site manifest.yaml --check")
sp("site manifest.yaml -o astro --no-charts")

In [ ]:
for path in sorted(Path("astro/src/content/docs/catalogue").rglob("*.mdx")):
    print(path.relative_to("astro"))

`--check` only validates the manifest and the files it lists, which is what a CI
job wants before a build. Without `--no-charts` the command also writes the chart
data under `astro/public/catalogue`. See the
[CLI reference](/scope-profiler/reference/cli/) for the manifest's other keys
(`executions` for nightly jobs, attached `files`, `images`) and for the sibling
`scope-profiler benchmark` command, which runs repeatable benchmarks and compares
their results.

## Where to go next

- [2. Post-processing](/scope-profiler/guide/02-postprocessing/) — the Python side of
  everything `inspect` prints.
- [3. Visualizing results](/scope-profiler/guide/03-visualization/) and
  [6. Building your own plots](/scope-profiler/guide/06-custom-plots/) — charts from code.
- [CLI reference](/scope-profiler/reference/cli/) — every flag of every command.